# F3 — Improved Models: RF, XGBoost & DistilBERT Fine-Tuning

**Objetivo**: Mejorar la clasificacion de sentimiento sobre el baseline de DistilBERT frozen + LogisticRegression (F3). Se prueban Random Forest, XGBoost sobre embeddings + engineered features, LoRA fine-tuning y full fine-tuning de DistilBERT.

**Metodologia**: Los modelos clasicos se entrenan sobre embeddings congelados de 768d combinados con features del EDA. El fine-tuning entrena DistilBERT directamente para clasificacion de secuencia.

**Salidas**: Tabla comparativa de metricas, graficos, tracking en MLflow, reporte JSON en `reports/`.


## 1. Montar Google Drive


In [ ]:
drive.mount('/content/drive')

DRIVE_BASE = "/content/drive/MyDrive"
PARQUET_PATH = f"{DRIVE_BASE}/office_products_balanced.parquet"
MLFLOW_PATH = f"{DRIVE_BASE}/mlruns"

print(f"Parquet: {PARQUET_PATH}")
print(f"MLflow:  {MLFLOW_PATH}")
print('Drive montado correctamente')


## 2. Instalar dependencias

Solo lo que no viene en Colab: `polars` para datos, `mlflow` para tracking, `xgboost` y `peft` para modelos.


In [ ]:
!pip install -q polars mlflow xgboost transformers -U
!pip install -q "torchao>=0.16.0" -U
!pip install -q peft -U


In [ ]:
import polars as pl
import numpy as np
import torch
import gc
import os
import json
import time
import mlflow
import shutil
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive
from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from transformers import (
    AutoTokenizer, AutoModel, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding,
    EarlyStoppingCallback
)
from peft import LoraConfig, get_peft_model, TaskType
from tqdm.notebook import tqdm


## 3. Cargar datos, muestreo y split

Misma estrategia que F3: muestra estratificada de 100k, split 70/15/15.


In [ ]:
SAMPLE_SIZE = 100_000
BATCH_SIZE = 256
MAX_LENGTH = 128
RANDOM_STATE = 42

ENG_FEATURES = [
    'mayusculas_count', 'char_total', 'exclamacion_count',
    'interrogacion_count', 'porcentaje_mayusculas',
    'puntuacion_emocional', 'total_tokens', 'unique_types', 'ttr'
]

df = pl.read_parquet(PARQUET_PATH)

dfs = []
for s in [0, 1, 2]:
    sub = df.filter(pl.col('sentiment') == s)
    n = min(SAMPLE_SIZE // 3, sub.shape[0])
    dfs.append(sub.sample(n=n, seed=RANDOM_STATE))

df_sample = pl.concat(dfs).sample(fraction=1.0, seed=RANDOM_STATE)
print(f"Sample: {df_sample.shape}")
print(df_sample['sentiment'].value_counts().sort('sentiment'))

texts = df_sample['text'].to_list()
labels = df_sample['sentiment'].to_numpy()
eng_features = df_sample.select(ENG_FEATURES).to_numpy()

indices = np.arange(len(texts))
train_idx, test_idx, _, _ = train_test_split(
    indices, labels, test_size=0.15, random_state=RANDOM_STATE, stratify=labels
)
train_idx_2, val_idx = train_test_split(
    train_idx, test_size=round(0.15/0.85, 3),
    random_state=RANDOM_STATE, stratify=labels[train_idx]
)

X_train = [texts[i] for i in train_idx_2]
X_val = [texts[i] for i in val_idx]
X_test = [texts[i] for i in test_idx]
y_train = labels[train_idx_2]
y_val = labels[val_idx]
y_test = labels[test_idx]

eng_train = eng_features[train_idx_2]
eng_val = eng_features[val_idx]
eng_test = eng_features[test_idx]

print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")


## 4. Extraer embeddings con DistilBERT (frozen)

Misma funcion que F3. Guardamos en Drive para no re-extraer si se corta la sesion.


In [ ]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()
print(f"Device: {device}")

def extract_embeddings(texts, model, tokenizer, batch_size=BATCH_SIZE, max_length=MAX_LENGTH):
    all_embeddings = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[i:i+batch_size]
        encoded = tokenizer(
            batch_texts, padding=True, truncation=True,
            max_length=max_length, return_tensors='pt'
        ).to(device)
        with torch.no_grad():
            outputs = model(**encoded)
        embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        all_embeddings.append(embeddings)
        del encoded, outputs
        if i % (batch_size * 10) == 0:
            gc.collect()
    return np.vstack(all_embeddings)


In [ ]:
print("Extrayendo embeddings TRAIN...")
X_train_emb = extract_embeddings(X_train, model, tokenizer)
print(f"Train embeddings: {X_train_emb.shape}")

print("Extrayendo embeddings VAL...")
X_val_emb = extract_embeddings(X_val, model, tokenizer)
print(f"Val embeddings: {X_val_emb.shape}")

print("Extrayendo embeddings TEST...")
X_test_emb = extract_embeddings(X_test, model, tokenizer)
print(f"Test embeddings: {X_test_emb.shape}")

np.save(f"{DRIVE_BASE}/train_embeddings.npy", X_train_emb)
np.save(f"{DRIVE_BASE}/val_embeddings.npy", X_val_emb)
np.save(f"{DRIVE_BASE}/test_embeddings.npy", X_test_emb)
print("Embeddings guardados en Drive")


## 5. Preparar features combinadas

Normalizamos las engineered features y las concatenamos con los embeddings para los modelos clasicos.


In [ ]:
scaler = StandardScaler()
eng_train_scaled = scaler.fit_transform(eng_train)
eng_val_scaled = scaler.transform(eng_val)
eng_test_scaled = scaler.transform(eng_test)

X_train_combined = np.concatenate([X_train_emb, eng_train_scaled], axis=1)
X_val_combined = np.concatenate([X_val_emb, eng_val_scaled], axis=1)
X_test_combined = np.concatenate([X_test_emb, eng_test_scaled], axis=1)

print(f"Combined dims - Train: {X_train_combined.shape}, Val: {X_val_combined.shape}, Test: {X_test_combined.shape}")


## 6. Random Forest

Grid search chico sobre n_estimators y max_depth.


In [ ]:
results = []

def eval_and_record(name, y_true, y_pred, training_time):
    metrics = {
        'model_name': name,
        'f1_macro': round(f1_score(y_true, y_pred, average='macro'), 4),
        'precision_macro': round(precision_score(y_true, y_pred, average='macro'), 4),
        'recall_macro': round(recall_score(y_true, y_pred, average='macro'), 4),
        'accuracy': round(accuracy_score(y_true, y_pred), 4),
        'training_time_seconds': round(training_time, 2),
        'confusion_matrix': confusion_matrix(y_true, y_pred).tolist(),
    }
    results.append(metrics)
    for k, v in metrics.items():
        if k != 'confusion_matrix':
            print(f"  {k}: {v}")


In [ ]:
print("=== Random Forest ===")
rf_params = {'n_estimators': [200], 'max_depth': [None], 'min_samples_leaf': [1, 4]}
best_f1 = -1

for params in ParameterGrid(rf_params):
    start = time.time()
    rf = RandomForestClassifier(**params, random_state=RANDOM_STATE, n_jobs=-1)
    rf.fit(X_train_combined, y_train)
    elapsed = time.time() - start
    val_pred = rf.predict(X_val_combined)
    f1 = f1_score(y_val, val_pred, average='macro')
    print(f"  params={params} -> val_f1={f1:.4f}, time={elapsed:.1f}s")
    if f1 > best_f1:
        best_f1 = f1
        y_pred_rf = rf.predict(X_test_combined)
        print("Random Forest - Test:")
        eval_and_record('Random Forest', y_test, y_pred_rf, elapsed)
    del rf
gc.collect()


## 7. XGBoost

Con early stopping sobre validation set para evitar overfitting.


In [ ]:
print("=== XGBoost ===")
xgb_params = {
    'n_estimators': 300,
    'max_depth': 6,
    'learning_rate': 0.1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'eval_metric': 'mlogloss',
    'use_label_encoder': False,
    'random_state': RANDOM_STATE,
    'early_stopping_rounds': 10,
}

start = time.time()
xgb = XGBClassifier(**xgb_params)
xgb.fit(
    X_train_combined, y_train,
    eval_set=[(X_val_combined, y_val)],
    verbose=False
)
elapsed = time.time() - start
y_pred_xgb = xgb.predict(X_test_combined)
print("XGBoost - Test:")
eval_and_record('XGBoost', y_test, y_pred_xgb, elapsed)
del xgb
gc.collect()


## 8. Fine-tuning DistilBERT

Entrenamos DistilBERT directamente para clasificacion de secuencia. Primero LoRA (param-efficient), luego full fine-tune.


### 8a. Preparar datasets

Tokenizamos y creamos Dataset objects para el Trainer de HuggingFace.


In [ ]:
from datasets import Dataset

def tokenize_fn(batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_LENGTH)

train_ds = Dataset.from_dict({'text': X_train, 'label': y_train})
val_ds = Dataset.from_dict({'text': X_val, 'label': y_val})
test_ds = Dataset.from_dict({'text': X_test, 'label': y_test})

train_ds = train_ds.map(tokenize_fn, batched=True)
val_ds = val_ds.map(tokenize_fn, batched=True)
test_ds = test_ds.map(tokenize_fn, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        'f1_macro': f1_score(labels, preds, average='macro'),
        'accuracy': accuracy_score(labels, preds),
    }


### 8b. LoRA Fine-Tuning

Congelamos el modelo base y entrenamos adapters LoRA en las capas de atencion.


In [ ]:
model_cls = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3
).to(device)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=['q_lin', 'k_lin', 'v_lin', 'out_lin']
)
model_lora = get_peft_model(model_cls, lora_config)
model_lora.print_trainable_parameters()


In [ ]:
lora_args = TrainingArguments(
    output_dir='/content/lora_checkpoints',
    eval_strategy='epoch',
    save_strategy='epoch',
    per_device_train_batch_size=128,
    per_device_eval_batch_size=256,
    num_train_epochs=3,
    learning_rate=2e-4,
    weight_decay=0.01,
    logging_dir='/content/logs_lora',
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    remove_unused_columns=False,
)

trainer_lora = Trainer(
    model=model_lora,
    args=lora_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

start = time.time()
trainer_lora.train()
elapsed = time.time() - start

preds_lora = trainer_lora.predict(test_ds)
y_pred_lora = np.argmax(preds_lora.predictions, axis=1)
print("LoRA - Test:")
eval_and_record('DistilBERT + LoRA', y_test, y_pred_lora, elapsed)
del model_lora, trainer_lora
gc.collect()
torch.cuda.empty_cache()


### 8c. Full Fine-Tune

Entrenamos todos los parametros de DistilBERT. Tarda ~3-4 horas en T4. Saltar si no hay tiempo.


In [ ]:
model_full = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3
).to(device)

full_args = TrainingArguments(
    output_dir='/content/full_checkpoints',
    eval_strategy='epoch',
    save_strategy='epoch',
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=4,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_dir='/content/logs_full',
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    remove_unused_columns=False,
)

trainer_full = Trainer(
    model=model_full,
    args=full_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

start = time.time()
trainer_full.train()
elapsed = time.time() - start

preds_full = trainer_full.predict(test_ds)
y_pred_full = np.argmax(preds_full.predictions, axis=1)
print("Full Fine-Tune - Test:")
eval_and_record('DistilBERT + Full FT', y_test, y_pred_full, elapsed)
del model_full, trainer_full
gc.collect()
torch.cuda.empty_cache()


## 9. Resultados comparativos

Graficos comparando todos los modelos: F1-macro, matrices de confusion.


In [ ]:
model_names = [r['model_name'] for r in results]
f1_scores = [r['f1_macro'] for r in results]
class_labels = ['Negativo', 'Neutro', 'Positivo']

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
colors = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6', '#f39c12']
bars = plt.barh(range(len(results)), f1_scores, color=colors[:len(results)])
plt.yticks(range(len(results)), model_names)
plt.xlabel('F1-macro')
plt.title('Comparacion de Modelos - F1-macro')
plt.xlim(0, 1)
for bar, val in zip(bars, f1_scores):
    plt.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
             f'{val:.4f}', va='center', fontsize=10)

n_models = len(results)
n_cols = min(3, n_models)
n_rows = (n_models + n_cols - 1) // n_cols

for i, r in enumerate(results):
    plt.subplot(n_rows, n_cols, i + 1)
    cm = np.array(r['confusion_matrix'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_labels, yticklabels=class_labels, cbar=False)
    plt.title(r['model_name'].split('+')[0].strip() if '+' in r['model_name'] else r['model_name'])
    plt.xlabel('Predicho')
    plt.ylabel('Real')

plt.tight_layout()
plt.show()


In [ ]:
print(f"{'Modelo':<40} {'F1-macro':<10} {'Precision':<10} {'Recall':<10} {'Accuracy':<10} {'Tiempo(s)':<10}")
print('-' * 90)
for r in results:
    print(f"{r['model_name']:<40} {r['f1_macro']:<10} {r['precision_macro']:<10} {r['recall_macro']:<10} {r['accuracy']:<10} {r['training_time_seconds']:<10}")


## 10. MLflow Tracking

Registramos todos los modelos en MLflow para tener el historial completo.


In [ ]:
mlflow.set_tracking_uri(f"file://{MLFLOW_PATH}")
mlflow.set_experiment("distilbert_improved")

for r in results:
    with mlflow.start_run(run_name=r['model_name']):
        mlflow.log_params({
            'model_name': r['model_name'],
            'sample_size': SAMPLE_SIZE,
            'batch_size': BATCH_SIZE,
            'max_length': MAX_LENGTH,
            'n_engineered_features': len(ENG_FEATURES),
        })
        mlflow.log_metrics({
            'f1_macro': r['f1_macro'],
            'precision_macro': r['precision_macro'],
            'recall_macro': r['recall_macro'],
            'accuracy': r['accuracy'],
            'training_time_seconds': r['training_time_seconds'],
        })

print(f"MLflow runs en: {MLFLOW_PATH}")


## 11. Exportar resultados

Exportamos metricas comparativas a JSON en reports/.


In [ ]:
os.makedirs('/content/reports', exist_ok=True)
export = {
    'baseline': 'DistilBERT frozen + LogisticRegression',
    'improved_results': [
        {k: v for k, v in r.items() if k != 'confusion_matrix'}
        for r in results
    ],
    'engineered_features': ENG_FEATURES,
    'sample_size': SAMPLE_SIZE,
    'best_model': max(results, key=lambda r: r['f1_macro']),
}

with open('/content/reports/metrics_distilbert_improved.json', 'w') as f:
    json.dump(export, f, indent=2)
print("Exportado: /content/reports/metrics_distilbert_improved.json")

if os.path.exists('/content/proyecto-integrador-ml-amazon-reviews'):
    shutil.copy(
        '/content/reports/metrics_distilbert_improved.json',
        '/content/proyecto-integrador-ml-amazon-reviews/reports/metrics_distilbert_improved.json'
    )
    print("Copiado a reports/ en el repo")


In [ ]:
del df, df_sample, model, tokenizer
del X_train_emb, X_val_emb, X_test_emb, y_train, y_val, y_test
del X_train_combined, X_val_combined, X_test_combined
del eng_train, eng_val, eng_test, eng_train_scaled, eng_val_scaled, eng_test_scaled
del texts, labels, eng_features, scaler
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Memoria liberada")
